In [1]:
# CELL 1: Install ML Prep Libraries
!pip install --upgrade scikit-learn ta

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29497 sha256=c12bccb8b2ee0223913b698e7df0197baf62ad832644b84dd17d4ff677e7c0cf
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\5f\67\4f\8a9f252836e053e532c6587a3230bc72a4deb16b03a829610b
Successfully built ta



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 1: Feature Engineering
import pandas as pd
import numpy as np
import ta

print("Loading Causal Features...")
df = pd.read_parquet('MTech_Causal_Features.parquet')

# Identify core columns
target_col = next(col for col in df.columns if 'Nifty' in col)
btc_col = next(col for col in df.columns if 'BTC' in col)
usdinr_col = next(col for col in df.columns if 'USDINR' in col)

print(f"Target: {target_col} | Causal Signal: {btc_col}, {usdinr_col}")

# 1. Build Pseudo_Price from BTC (24/7 continuous data for stable indicators)
df['Pseudo_Price'] = np.exp(df[btc_col].cumsum()) * 10000

# 2. Add technical indicators for context
df['RSI']      = ta.momentum.RSIIndicator(close=df['Pseudo_Price'], window=14).rsi()
df['MACD']     = ta.trend.MACD(close=df['Pseudo_Price']).macd_diff()
df['BB_Width'] = ta.volatility.BollingerBands(close=df['Pseudo_Price']).bollinger_wband()

# 3. Add causal lag features (Giving the AI 'memory' of the lead-lag relationship)
df['USDINR_Lag15'] = df[usdinr_col].shift(15)
df['BTC_Lag3']     = df[btc_col].shift(3)

# 4. Clean up
df = df.drop(columns=['Pseudo_Price']).dropna()

print(f" Feature Engineering Complete. Shape: {df.shape}")
print(df.head(3))

Loading Causal Features...
Target: Nifty_Close_Return | Causal Signal: BTC_Close_Return, USDINR_Close_Return
 Feature Engineering Complete. Shape: (36310, 8)
                           USDINR_Close_Return  BTC_Close_Return  \
Datetime                                                           
2026-03-19 14:04:00+00:00            -0.000183         -0.001086   
2026-03-19 14:05:00+00:00             0.000107         -0.000232   
2026-03-19 14:06:00+00:00            -0.000258         -0.000617   

                           Nifty_Close_Return        RSI      MACD  BB_Width  \
Datetime                                                                       
2026-03-19 14:04:00+00:00                 0.0  50.886624 -4.267670  0.832382   
2026-03-19 14:05:00+00:00                 0.0  50.092847 -5.186027  0.770433   
2026-03-19 14:06:00+00:00                 0.0  47.952670 -6.040645  0.682365   

                           USDINR_Lag15  BTC_Lag3  
Datetime                                        

In [3]:
# CELL 2: Splitting and Scaling
from sklearn.preprocessing import StandardScaler
import pickle

# 1. Chronological Split (80% Train, 20% Test)
split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx].copy()
test_df   = df.iloc[split_idx:].copy()

print(f"Training set : {len(train_df):,} rows")
print(f"Testing set  : {len(test_df):,} rows")

# 2. Scaling (Fit only on Train)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df)
train_scaled_df = pd.DataFrame(train_scaled, columns=df.columns, index=train_df.index)

test_scaled = scaler.transform(test_df)
test_scaled_df = pd.DataFrame(test_scaled, columns=df.columns, index=test_df.index)

# Save scaler for future use
with open('MTech_Scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(" Scaled features prepared. Stored both Scaled and Raw Dataframes.")

Training set : 29,048 rows
Testing set  : 7,262 rows
 Scaled features prepared. Stored both Scaled and Raw Dataframes.


In [4]:
# CELL 3: 3D Sequence Generation
import numpy as np

def create_sequences(scaled_df, raw_df, window_size=12):
    """
    X = Scaled features (Z-scores) for the Neural Net
    y = Raw returns (Percentages) for the Backtest/Reward
    """
    X_data = scaled_df.values
    raw_data = raw_df.values
    
    # Identify index for Nifty target
    nifty_col = next(col for col in scaled_df.columns if 'Nifty' in col)
    target_idx = scaled_df.columns.get_loc(nifty_col)

    X, y = [], []
    for i in range(window_size, len(X_data)):
        # X gets the 12-minute window of SCALED features
        X.append(X_data[i-window_size:i, :])
        
        # y gets the SINGLE next-step RAW return (the percentage move)
        y.append(raw_data[i, target_idx])

    return np.array(X), np.array(y)

window_size = 12 

#  Apply the Dual-Stream fix
X_train, y_train = create_sequences(train_scaled_df, train_df, window_size)
X_test,  y_test  = create_sequences(test_scaled_df,  test_df,  window_size)

print(f"\n DAY 3 COMPLETE!")
print(f"X_train Shape: {X_train.shape} (Scaled Inputs)")
print(f"y_train Range: {y_train.min():.6f} to {y_train.max():.6f} (Raw Returns)")

# Save tensors
np.savez_compressed(
    'MTech_Tensors.npz',
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test
)
print("\n Saved Fixed 3D Tensors → 'MTech_Tensors.npz'")


 DAY 3 COMPLETE!
X_train Shape: (29036, 12, 8) (Scaled Inputs)
y_train Range: -0.021322 to 0.030413 (Raw Returns)

 Saved Fixed 3D Tensors → 'MTech_Tensors.npz'


In [7]:
# NEW CELL: Overcoming Pipeline Limitations (Prevents Target Leakage & Allows Cross-Country Swapping)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Define the fully generalized, leakage-protected engineering function
def engineer_invariant_signals(df_input, target_asset_prefix='Nifty'):
    """
    Engineers globally invariant features while completely isolating the target return 
    from the feature state space to guarantee zero lookahead bias.
    """
    df_out = pd.DataFrame(index=df_input.index)
    
    # LIMITATION OVERCOME: Dynamically find the target based on prefix (e.g., 'Nifty', 'SP500', 'FTSE')
    target_return_col = next(col for col in df_input.columns if target_asset_prefix in col)
    
    # Reconstruct a local pseudo-price scale from raw returns for technical metrics
    pseudo_price = np.exp(df_input[target_return_col].cumsum()) * 10000
    
    # 8 Globally Invariant Features (State Space Inputs)
    df_out['log_return']   = np.log(pseudo_price / pseudo_price.shift(1))
    df_out['realized_vol'] = df_out['log_return'].rolling(window=5).std()
    
    delta = pseudo_price.diff()
    gain  = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs    = gain / (loss + 1e-9)
    df_out['rsi'] = 100 - (100 / (1 + rs))
    
    ema_12 = pseudo_price.ewm(span=12, adjust=False).mean()
    ema_26 = pseudo_price.ewm(span=26, adjust=False).mean()
    macd   = ema_12 - ema_26
    signal = macd.ewm(span=9, adjust=False).mean()
    df_out['macd_hist'] = macd - signal
    
    ma_20  = pseudo_price.rolling(window=20).mean()
    std_20 = pseudo_price.rolling(window=20).std()
    df_out['bb_width']  = (4 * std_20) / ma_20
    
    df_out['return_lag_1'] = df_out['log_return'].shift(1)
    
    # Preserving cross-market features safely
    if 'USDINR_Lag15' in df_input.columns:
        df_out['USDINR_Lag15'] = df_input['USDINR_Lag15']
    if 'BTC_Lag3' in df_input.columns:
        df_out['BTC_Lag3'] = df_input['BTC_Lag3']
        
    # Clean up NaNs from sliding window indicators
    clean_features = df_out.dropna()
    
    # LIMITATION OVERCOME: Capture target returns strictly shifted forward by 1 step 
    # to serve exclusively as the RL environment's reward channel.
    raw_targets = df_input.loc[clean_features.index, target_return_col].shift(-1).dropna()
    final_features = clean_features.loc[raw_targets.index]
    
    # CRITICAL: Strip out any raw return tracks of the target from the final features list 
    # if they accidentally slipped in, ensuring zero target leakage.
    final_features = final_features.drop(columns=[target_return_col], errors='ignore')
    
    return final_features, raw_targets

# 2. Run the updated pipeline (Change 'Nifty' to any asset to generate its unique tensors)
invariant_features, continuous_targets = engineer_invariant_signals(df, target_asset_prefix='Nifty')

# 3. Chronological Train/Test Split
split_idx = int(len(invariant_features) * 0.8)
train_features = invariant_features.iloc[:split_idx]
test_features  = invariant_features.iloc[split_idx:]
train_targets  = continuous_targets.iloc[:split_idx]
test_targets   = continuous_targets.iloc[split_idx:]

# 4. Localized Domain Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_features)
X_test_scaled  = scaler.transform(test_features)

# 5. Build Sequential 3D Tensors
def build_sequential_tensors(scaled_matrix, target_series, lookback=12):
    X, y = [], []
    target_values = target_series.values
    for i in range(lookback, len(scaled_matrix)):
        X.append(scaled_matrix[i-lookback:i])
        y.append(target_values[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train_final, y_train_final = build_sequential_tensors(X_train_scaled, train_targets)
X_test_final,  y_test_final  = build_sequential_tensors(X_test_scaled,  test_targets)

# 6. Compress and Save over old matrices
np.savez_compressed(
    'MTech_Tensors.npz',
    X_train=X_train_final, y_train=y_train_final,
    X_test=X_test_final,   y_test=y_test_final
)

print("🚀 ALL PIPELINE LIMITATIONS OVERCOME!")
print(f"Features dimension count: {X_train_final.shape[2]} (Target safely removed from input array)")
print(f"Train Shape: {X_train_final.shape} | Test Shape: {X_test_final.shape}")

🚀 ALL PIPELINE LIMITATIONS OVERCOME!
Features dimension count: 8 (Target safely removed from input array)
Train Shape: (29020, 12, 8) | Test Shape: (7246, 12, 8)
